# 업무 안내서 문장 유사도 분석 (30분 실습)
### 질문하면 안내서에서 근거 문장을 찾아 준다

**환경:** Google Colab (인터넷 연결 필요)
**준비물:** 개인정보보호위원회 「개인정보 보호 교육 업무 안내서」 PDF (2024.12, 29쪽)
**소요 시간:** 약 30분

## 오늘의 상황
> 신입 개인정보 담당자가 "교육은 1년에 몇 번 해야 하나요?"라고 묻습니다.
> 29쪽짜리 안내서를 처음부터 읽는 대신, **질문과 뜻이 비슷한 문장**을 찾아 쪽 번호와 함께 보여 줍니다.

## 실습 진행 과정
| 시간 | 단계 | 할 일 | 산출물 |
|---|---|---|---|
| 0–3분 | 0. 준비 | 설치, PDF 올리기 | — |
| 3–13분 | 1. 텍스트 전처리 | PDF에서 글자 뽑기 → 머리글·표 지우기 → 글머리 기호로 문장 나누기 | 문장 목록 |
| 13–16분 | 2. 정제 데이터셋 | 표로 정리, 짧은 조각 빼기, CSV 저장 | `manual_sentences.csv` |
| 16–21분 | 3. 텍스트 분석 | 문장 임베딩 모델 점검 → 모든 문장을 벡터로 | 문장 벡터 |
| 21–28분 | 4. 분석 결과 | 질문 검색, 비슷한 문장 쌍, 정답 세트로 평가 | 결과 표, CSV 2개 |
| 28–30분 | 정리 | 한계 | — |

> **실행 방법:** 셀을 클릭하고 `Shift + Enter`. 반드시 **위에서부터 순서대로** 실행하세요.
> ✏️ 표시가 있는 곳은 **직접 바꿔 볼 수 있는 부분**입니다.

> 📌 **출처:** 개인정보보호위원회, 「개인정보 보호 교육 업무 안내서」 2024.12.
> 안내서에 "무단전재를 금하며, 가공·인용할 때는 출처를 밝혀 주시기 바랍니다"라고 적혀 있으므로, 이 실습에서 만드는 데이터셋에도 출처를 함께 적습니다.

---
## 0. 준비 (3분)
| 도구 | 하는 일 |
|---|---|
| `pdfplumber` | PDF에서 글자를 줄 단위로 뽑기 (한국어 줄 구조가 잘 유지됨) |
| `sentence-transformers` | 사전학습 **문장 임베딩** 모델 사용 |

In [ ]:
# pip: 파이썬 도구 설치 명령, 맨 앞의 !는 파이썬이 아닌 터미널 명령으로 실행하라는 표시
# -q: 설치 과정 메시지를 줄여서 출력
!pip install -q pdfplumber sentence-transformers

Colab 왼쪽 **📁 파일 창**에 안내서 PDF 파일을 끌어다 놓으세요. 아래 셀이 PDF 파일을 **자동으로 찾습니다.**

In [ ]:
# glob: 파일 이름을 패턴으로 찾는 파이썬 기본 도구
import glob

# glob.glob("*.pdf"): 현재 폴더에서 이름이 .pdf로 끝나는 파일들의 경로를 리스트로 돌려줌
#   (*는 '아무 글자나 몇 글자든'이라는 뜻)
# [0]: 그 리스트의 첫 번째 파일을 사용 (PDF가 없으면 리스트가 비어 IndexError 발생)
pdf_path = glob.glob("*.pdf")[0]

# 찾은 파일 이름 출력
print("PDF 파일:", pdf_path)

---
## 1. 텍스트 데이터 전처리 (10분)

### 💡 이 안내서의 특징
| 특징 | 예 | 처리 방법 |
|---|---|---|
| 쪽마다 반복되는 **머리글** | `8 개인정보 보호 교육 업무 안내서` | 정규표현식으로 지우기 |
| 장 번호만 있는 줄 | `Ⅱ` | 지우기 |
| **개조식 문장** (`~하여야 함`, `~가능`) | `⬛ 개인정보처리자는 … 연 1회 이상 교육을 수행하여야 함` | 마침표가 없어서 **글머리 기호**(⬛ ⚫ ※ • - ① 가. 1.)로 문장을 나눔 |
| 한 문장이 여러 줄로 나뉨 | `… 교육을` ↵ `수행하여야 함` | 글머리 기호가 없는 줄은 **앞줄에 이어 붙이기** |
| **표**와 그림 | 교육 절차 그림, 점검표 | 표 영역을 잘라내고 나머지 글자만 뽑기 |
| 소제목 | `참고 ┃ …`, `2 적용 대상` | 문장에 붙지 않도록 끊기 |

### 💡 분석할 쪽
- 7쪽(용어의 정의) ~ 17쪽(사후 관리)만 씁니다. (PDF 뷰어에 보이는 쪽 번호 기준)
- 1~6쪽은 표지·목차·발간 목적, 18~19쪽은 표가 대부분이고, 20쪽부터는 빈칸 서식입니다.

In [ ]:
# pdfplumber: PDF에서 글자를 뽑는 도구
# re: 정규표현식(글자 패턴) 도구
import pdfplumber
import re

# ===== ✏️ [입력] 분석할 쪽 범위 (PDF 쪽 번호, 끝 쪽 포함) =====
PAGE_START, PAGE_END = 7, 17
# ===========================================================

# 문장이 시작되는 글머리 기호 패턴
#   ^ : 줄의 맨 앞
#   ⬛ ⚫ ※ • : 안내서에서 쓰는 글머리 기호
#   "- "  : 하이픈 + 띄어쓰기
#   [①-⑳] : 동그라미 숫자 ①~⑳ 중 하나
#   \d+\) : 1) 2) 처럼 숫자 + 닫는 괄호
#   [가-하]\. : 가. 나. 다. 처럼 한글 한 글자 + 마침표 + 띄어쓰기
#   \d+\. : 1. 2. 처럼 숫자 + 마침표 + 띄어쓰기
BULLET = re.compile(r"^(⬛|⚫|※|•|- |[①-⑳]|\d+\)|[가-하]\. |\d+\. )")

# 통째로 버릴 줄 패턴: 쪽 머리글('8 개인정보 보호 교육 업무 안내서')과 장 번호만 있는 줄('Ⅱ')
SKIP = re.compile(r"^\d+\s*개인정보 보호 교육 업무 안내서$|^[ⅠⅡⅢⅣⅤ]$")

# 소제목 패턴: '참고 ┃ ...' 상자 제목, '2 적용 대상' 같은 번호 소제목, '…법 제28조(…)' 같은 법조문 상자 제목
HEADING = re.compile(r"^참고\s*┃|^\d+\s+[가-힣]|제\d+조\(.+\)$")

# 문장 단위 결과를 [쪽 번호, 원문] 형태로 모을 빈 리스트
units = []

# with 문: PDF 파일을 열고, 블록이 끝나면 자동으로 닫음
with pdfplumber.open(pdf_path) as pdf:
    # 분석할 쪽 번호를 하나씩 꺼냄 (range의 끝 값은 포함되지 않으므로 +1)
    for page_no in range(PAGE_START, PAGE_END + 1):
        # pdf.pages는 0부터 번호가 매겨지므로 쪽 번호 - 1로 해당 쪽을 꺼냄
        page = pdf.pages[page_no - 1]

        # page.find_tables(): 이 쪽에서 표로 인식된 영역들의 목록
        # page.outside_bbox(영역): 그 영역 바깥 부분만 남긴 새 page 객체를 돌려줌 → 표 글자를 제거
        for table in page.find_tables():
            page = page.outside_bbox(table.bbox)

        # 지금 만들고 있는 문장 ([쪽 번호, 글자]), 아직 없으면 None
        cur = None

        # page.extract_text(): 이 쪽의 글자를 줄바꿈(\n)이 들어간 문자열 하나로 돌려줌
        # .splitlines(): 줄 단위 리스트로 나눔
        for line in page.extract_text().splitlines():
            # 줄 앞뒤의 공백 제거
            line = line.strip()

            # 빈 줄이거나 버릴 줄이면 다음 줄로 넘어감
            if not line or SKIP.match(line):
                continue

            # 글머리 기호로 시작하거나 소제목이면: 지금까지 만든 문장을 저장하고 새로 시작
            if BULLET.match(line) or HEADING.match(line):
                if cur:
                    units.append(cur)
                # 글머리 기호면 새 문장을 시작, 소제목이면 문장을 시작하지 않음(None)
                cur = [page_no, line] if BULLET.match(line) else None

            # 글머리 기호가 없는 줄이고 만들던 문장이 있으면: 앞 문장 뒤에 띄어쓰기 한 칸 두고 이어 붙임
            elif cur:
                cur[1] += " " + line

        # 쪽이 끝나면 마지막으로 만들던 문장도 저장
        if cur:
            units.append(cur)

# 나뉜 문장 수와 앞 5개 출력
print("나뉜 문장 수:", len(units))
for page_no, text in units[:5]:
    print(page_no, "|", text[:80])

✅ **결과 확인:** `⬛`, `⚫`, `가.` 같은 기호로 시작하는 문장이 쪽 번호와 함께 나왔나요?

> ⚠️ PDF는 줄 끝에서 단어가 잘리는 경우가 있어(`정정` → `정 정`, `업무` → `업 무`) 이어 붙인 문장에 띄어쓰기가 잘못 들어간 곳이 있습니다. 임베딩 모델은 이런 작은 띄어쓰기 차이에 비교적 덜 민감하므로 이번 실습에서는 그대로 둡니다.

---
## 2. 정제 데이터셋 만들기 (3분)
문장 목록을 **표(DataFrame)** 로 만들고 다음을 정리합니다.
- 맨 앞의 글머리 기호(`⬛`, `가.` 등) 지우기 → `문장` 열
- 20자 미만 조각(소제목 등) 빼기
- 문장 번호(`문장ID`)와 출처 붙이기

In [ ]:
# pandas: 표(DataFrame) 다루기 도구
import pandas as pd

# [쪽 번호, 원문] 리스트를 두 열짜리 표로 변환
df = pd.DataFrame(units, columns=["쪽", "원문"])

# .str.replace(패턴, "", regex=True): 각 원문 맨 앞의 글머리 기호를 빈 문자열로 바꿈(= 지움)
# .str.strip(): 앞뒤 공백 제거
df["문장"] = df["원문"].str.replace(BULLET, "", regex=True).str.strip()

# .str.len(): 문장의 글자 수
df["글자수"] = df["문장"].str.len()

# 글자 수가 20자 이상인 행만 남기고, 행 번호를 0부터 다시 매김
df = df[df["글자수"] >= 20].reset_index(drop=True)

# 문장ID: S000, S001 … 형식의 번호 (:03d는 세 자리 숫자로 앞을 0으로 채우는 형식)
df.insert(0, "문장ID", [f"S{i:03d}" for i in range(len(df))])

# 출처: 가공한 데이터에도 원문 출처를 남김
df["출처"] = "개인정보보호위원회, 「개인정보 보호 교육 업무 안내서」 2024.12."

# 정제 데이터셋을 CSV로 저장 (index=False: 행 번호 저장 안 함, utf-8-sig: 엑셀에서 한글 깨짐 방지)
df.to_csv("manual_sentences.csv", index=False, encoding="utf-8-sig")

# 문장 수, 쪽별 문장 수, 글자 수 요약 출력
print("정제된 문장 수:", len(df))
print("쪽별 문장 수:", df["쪽"].value_counts().sort_index().to_dict())
print("글자 수 (최소/평균/최대):", df["글자수"].min(), round(df["글자수"].mean()), df["글자수"].max())

# 표의 앞 5행 보기 (원문 열은 빼고 출력)
df[["문장ID", "쪽", "문장", "글자수"]].head()

✅ **결과 확인:** 📁 파일 창에 `manual_sentences.csv`가 생겼나요? 내려받아 엑셀로 열어 보세요.

| 열 | 뜻 |
|---|---|
| `문장ID` | 문장 번호 |
| `쪽` | PDF 쪽 번호 (근거 위치) |
| `원문` | 글머리 기호가 붙은 원래 문장 |
| `문장` | 분석에 쓰는 정제된 문장 |
| `글자수` | 문장 길이 |
| `출처` | 원문 출처 |

---
## 3. 텍스트 분석: 문장 임베딩 (5분)
**문장 임베딩**은 문장을 "뜻이 담긴 숫자 목록(벡터)"으로 바꿉니다. 뜻이 비슷한 문장은 벡터도 가까워집니다.

| 항목 | 내용 |
|---|---|
| 모델 | `jhgan/ko-sroberta-multitask` (한국어 문장 임베딩 모델) |
| 결과 | 문장 하나 → 숫자 768개 |

### 💡 먼저 모델을 점검합니다
모델이 제대로 불러와지지 않으면 **모든 문장의 유사도가 0.9 이상으로 똑같이** 나와 검색이 되지 않습니다.
그래서 **뜻이 비슷한 쌍**과 **관계없는 쌍**의 유사도를 먼저 비교합니다.

> ⚠️ 처음 실행할 때 모델을 내려받느라 1~3분 걸릴 수 있습니다.

In [ ]:
# SentenceTransformer: 사전학습 문장 임베딩 모델을 불러오는 클래스
# cosine_similarity: 벡터들 사이의 코사인 유사도(-1 ~ 1, 1에 가까울수록 방향이 같음)를 계산하는 함수
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Hugging Face Hub에서 모델 파일을 내려받아 불러옴 (한 번 내려받으면 캐시에 저장됨)
model = SentenceTransformer("jhgan/ko-sroberta-multitask")

# model.encode(문장 리스트): 문장마다 벡터를 만들어 (문장 수 × 768) 크기의 배열로 돌려줌
check = model.encode(["교육은 매년 한 번 이상 해야 합니다",       # 0번: 기준 문장
                      "연 1회 이상 교육을 실시하여야 함",          # 1번: 0번과 뜻이 비슷함
                      "오늘 점심으로 김밥을 먹었습니다"])          # 2번: 0번과 관계없음

# cosine_similarity(check): 3 × 3 유사도 표, [0, 1]은 0번-1번, [0, 2]는 0번-2번 문장의 유사도
sim = cosine_similarity(check)
print(f"비슷한 쌍: {sim[0, 1]:.2f} / 관계없는 쌍: {sim[0, 2]:.2f}")

# 비슷한 쌍이 관계없는 쌍보다 0.1 이상 높으면 정상으로 판단
print("벡터 칸 수:", check.shape[1])
print("모델 점검:", "정상" if sim[0, 1] - sim[0, 2] >= 0.1 else "⚠️ 비정상 — 아래 '문제가 생기면'을 보세요")

> 🔧 **문제가 생기면:** "비정상"이 나오면 새 셀에서 `!pip install -q -U sentence-transformers transformers`를 실행하고,
> 메뉴 **런타임 → 세션 다시 시작** 후 처음부터 다시 실행하세요.

In [ ]:
# 정제된 모든 문장을 벡터로 변환 (show_progress_bar=True: 진행 막대 표시)
# 결과 X: (문장 수 × 768) 크기의 배열, X의 i번째 행 = df의 i번째 문장의 벡터
X = model.encode(df["문장"].tolist(), show_progress_bar=True)

# X.shape: (행 수, 열 수) = (문장 수, 벡터 칸 수)
print("문장 벡터 크기:", X.shape)

---
## 4. 분석 결과 (7분)

### 4-1. 질문으로 근거 문장 찾기
질문도 같은 모델로 벡터로 바꾼 뒤, 모든 문장과의 코사인 유사도를 계산해 **가장 높은 5개**를 보여 줍니다.

In [ ]:
# 질문 하나를 받아 유사도가 높은 문장 k개를 표로 돌려주는 함수
def search(query, k=5):
    # 질문을 리스트로 감싸 넣어 (1 × 768) 배열로 변환
    q = model.encode([query])

    # 질문과 모든 문장의 유사도: (1 × 문장 수) 배열 → [0]으로 1차원 배열로 꺼냄
    scores = cosine_similarity(q, X)[0]

    # argsort(): 작은 값부터의 위치 순서 → [::-1]로 뒤집어 큰 값부터 → [:k]로 앞 k개
    top = scores.argsort()[::-1][:k]

    # 해당 행들의 문장ID·쪽·문장을 꺼내고, 유사도 열을 소수 셋째 자리까지 붙여 돌려줌
    return df.loc[top, ["문장ID", "쪽", "문장"]].assign(유사도=scores[top].round(3))

# ===== ✏️ [입력] 질문 =====
query = "개인정보 보호 교육은 1년에 몇 번 해야 하나요?"
# =========================

# 검색 결과 표 출력
search(query)

**✏️ [입력]** `query`를 바꿔 보세요. 예) `아르바이트생도 교육 대상인가요?`, `교육 증빙은 무엇으로 남기나요?`, `무료 교육 업체를 조심해야 하는 이유는?`

### 4-2. 서로 비슷한 문장 쌍 찾기
모든 문장끼리 유사도를 계산해 **가장 비슷한 쌍 10개**를 찾습니다. 업무 문서에서 **같은 내용이 반복된 곳**을 찾을 때 쓸 수 있습니다.

In [ ]:
# 모든 문장끼리의 유사도 표: (문장 수 × 문장 수), sim[i, j] = i번 문장과 j번 문장의 유사도
sim = cosine_similarity(X)

# i < j인 모든 쌍을 (유사도, i, j) 튜플로 모음 (같은 쌍을 두 번 세지 않고, 자기 자신과의 쌍은 제외)
pairs = [(sim[i, j], i, j) for i in range(len(df)) for j in range(i + 1, len(df))]

# 유사도가 큰 순서로 정렬해서 앞 10쌍을 표로 만듦
top_pairs = pd.DataFrame(
    [{"유사도": round(s, 3),
      "문장1": f"[{df['쪽'][i]}쪽] {df['문장'][i]}",
      "문장2": f"[{df['쪽'][j]}쪽] {df['문장'][j]}"} for s, i, j in sorted(pairs, reverse=True)[:10]]
)

# CSV로 저장하고 표 출력
top_pairs.to_csv("similar_pairs.csv", index=False, encoding="utf-8-sig")
top_pairs

✅ **결과 확인:** 서로 다른 쪽에서 같은 내용을 반복한 문장 쌍이 보이나요?
예를 들어 7쪽 용어 정의의 "개인정보취급자란 … 임직원, 파견근로자, 시간제 근로자 등을 말한다"와 14쪽 교육 대상 설명의 "… 임직원, 파견근로자, 시간제 근로자 등을 말함"은 거의 같은 문장입니다. 이런 쌍이 위에 나오는지 확인해 보세요.

### 4-3. 정답 세트로 평가하기
지금까지는 결과를 눈으로만 봤습니다. 이번에는 **질문 10개와 정답**을 미리 정해 두고, 상위 5개 안에 정답 문장이 들어오는 비율(**Hit@5**)을 계산합니다.

- **정답 표현:** 정답 문장에만 들어 있는 표현입니다. 상위 5개 문장 중 하나라도 이 표현을 포함하면 맞힌 것으로 봅니다.
- `|`는 "또는"이라는 뜻입니다. (예: `근로형태|시간제` = 근로형태 또는 시간제)
- 질문은 일부러 **안내서와 다른 말**로 썼습니다. (예: 안내서 `연 1회` ↔ 질문 `1년에 몇 번`)

In [ ]:
# ===== ✏️ [입력] 정답 세트: (질문, 정답 문장에 들어 있는 표현) =====
GOLD = [
    ("개인정보 보호 교육은 1년에 몇 번 해야 하나요?",          "연 1회"),
    ("한 번 교육할 때 최소 몇 시간 정도 해야 하나요?",          "1시간"),
    ("교육 결과 자료는 얼마 동안 보관해야 하나요?",              "1년 이상 보관"),
    ("아르바이트나 계약직 직원도 교육을 받아야 하나요?",          "근로형태|시간제"),
    ("일을 맡긴 외부 업체 직원도 교육해야 하나요?",              "수탁자를 교육|교육대상에 포함|수탁자에 대한 교육"),
    ("출장 때문에 교육에 못 간 직원은 어떻게 하나요?",            "불참자"),
    ("직원이 교육을 들었다는 증거는 무엇으로 남기나요?",          "증빙자료"),
    ("교육을 공짜로 해 준다는 업체를 조심해야 하는 이유는?",      "금융상품"),
    ("교육 계획은 누구에게 허락을 받아야 하나요?",               "승인"),
    ("교육이 끝난 뒤 효과는 어떻게 확인하나요?",                 "효과성"),
]
# ================================================================

# 질문마다 결과를 담을 빈 리스트
rows = []

# 질문과 정답 표현을 하나씩 꺼냄
for question, answer in GOLD:
    # 질문으로 상위 5개 문장 검색
    result = search(question, k=5)

    # .str.contains(answer): 각 문장이 정답 표현(정규표현식)을 포함하는지 True/False
    # .any(): 5개 중 하나라도 True면 True
    hit = result["문장"].str.contains(answer).any()

    # 정답 문장 수: 전체 데이터셋에서 정답 표현을 포함한 문장 수 (0이면 정답 세트가 잘못된 것)
    rows.append({"질문": question, "정답 표현": answer,
                 "정답 문장 수": df["문장"].str.contains(answer).sum(),
                 "1위 문장": result["문장"].iloc[0][:40],
                 "맞힘": hit})

# 질문별 결과 표
evaluation = pd.DataFrame(rows)

# Hit@5 = 맞힌 질문 수 ÷ 전체 질문 수 (True는 1, False는 0으로 계산되므로 평균이 곧 비율)
print(f"Hit@5: {evaluation['맞힘'].mean():.0%} ({evaluation['맞힘'].sum()} / {len(evaluation)})")

# CSV로 저장하고 표 출력
evaluation.to_csv("evaluation.csv", index=False, encoding="utf-8-sig")
evaluation

✅ **결과 확인**
- `정답 문장 수`가 0인 질문이 있으면 전처리에서 그 문장이 빠졌거나 정답 표현이 잘못된 것입니다.
- 틀린 질문의 `1위 문장`을 읽어 보세요. 왜 그 문장이 1위가 되었을까요?

---
## 정리 (2분)

### 🗂️ 만든 파일
| 파일 | 내용 |
|---|---|
| `manual_sentences.csv` | 정제 데이터셋 (문장ID, 쪽, 원문, 문장, 글자수, 출처) |
| `similar_pairs.csv` | 가장 비슷한 문장 쌍 10개 |
| `evaluation.csv` | 정답 세트 평가 결과 |

### 🤔 생각해 볼 점 (한계)
| 한계 | 예 | 더 나은 방법 |
|---|---|---|
| **표 내용이 빠짐** | 교육 절차 그림, 부적절한 교육 유형 표 | 표를 칸 단위로 뽑아 문장으로 바꾸기 |
| **문장 단위는 사람이 정한 규칙** | 글머리 기호 하나 = 문장 하나 | 문서마다 규칙을 조정 |
| **정답 세트가 작음** | 질문 10개 → 한 문제가 10% | 질문을 늘리고 여러 사람이 정답 확인 |
| **Hit@5는 넉넉한 기준** | 정답이 5위여도 맞힘 | 1위 정답률, 정답 순위도 함께 보기 |
| **유사도 기준값이 없음** | 0.6이 높은지 낮은지 모델마다 다름 | 순위로 보고 직접 확인 |
| **최신성** | 안내서는 2024.12. 기준 | 법령은 국가법령정보센터에서 최신 내용 확인 |

**참고:** [pdfplumber](https://github.com/jsvine/pdfplumber) · [sentence-transformers](https://www.sbert.net/) · [ko-sroberta-multitask](https://huggingface.co/jhgan/ko-sroberta-multitask) · [개인정보보호위원회](https://www.pipc.go.kr)